# 🏠 House Price Prediction — Exploratory Data Analysis

## Problem Statement

Real estate pricing is influenced by a complex mix of structural, 
locational, and conditional factors. The goal of this project is to 
build a regression model that accurately predicts the sale price of 
residential homes in Ames, Iowa based on 80 features describing 
various aspects of the property.

The dataset is sourced from the Kaggle competition:
"House Prices: Advanced Regression Techniques"

## Agenda

1. **Import Libraries and Load Data**
2. **Initial Data Inspection** — shape, dtypes, head, describe
3. **Target Variable Analysis** — distribution of SalePrice
4. **Missing Value Analysis** — identify and strategize handling
5. **Numerical Feature Analysis** — distributions, outliers, correlation with SalePrice
6. **Categorical Feature Analysis** — cardinality, frequency, impact on SalePrice
7. **Feature Relationships** — multicollinearity, pairplots, heatmap
8. **Key Observations and Next Steps**

In [8]:
#importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_profiling import ProfileReport #useful library which gives a summary report of the data

In [24]:
df=pd.read_csv('Data/train.csv')
df.sample(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
870,871,20,RL,60.0,6600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2009,WD,Normal,109500
1040,1041,20,RL,88.0,13125,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,NaN,0,1,2006,WD,Normal,155000
243,244,160,RL,75.0,10762,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2009,WD,Normal,120000
53,54,20,RL,68.0,50271,Pave,NaN,IR1,Low,AllPub,...,0,NaN,NaN,NaN,0,11,2006,WD,Normal,385000
327,328,20,RL,80.0,11600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,145250


In [25]:
df.shape

(1460, 81)

1460 rows and 81 columns or 80 features

That's alot of features to work on, so I will drop the features with many NaN values

In [26]:
df.isnull().sum()[df.isnull().sum()>0]

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

### Dropping Features
Dropping columns: Alley, MasVnrType, FireplaceQu, PoolQC, Fence, MiscFeature as imputing them is a hassle and would introduce noise

In [29]:
df=df.drop(['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature'],axis=1,)
df.sample(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
635,636,190,RH,60.0,10896,Pave,Reg,Bnk,AllPub,Inside,...,0,0,0,0,0,3,2007,WD,Abnorml,200000
1280,1281,20,RL,67.0,9808,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,3,2009,WD,Normal,227000
1002,1003,20,RL,75.0,11957,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,7,2008,WD,Normal,232000
65,66,60,RL,76.0,9591,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,10,2007,WD,Normal,317000
900,901,20,RL,NaN,7340,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,6,2007,WD,Normal,110000


In [32]:
#checking the datatypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 75 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Neighborhood   1460 non-null   object 
 12  Condition1     1460 non-null   object 
 13  Condition2     1460 non-null   object 
 14  BldgType       1460 non-null   object 
 15  HouseStyle     1460 non-null   object 
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuil

### Data Types
The dataset contains a mix of numerical and categorical features. 
Categorical features are stored as `object` dtype, which cannot be 
directly consumed by ML models. These will need to be handled via 
encoding techniques (Label Encoding or One-Hot Encoding) during 
the preprocessing stage.

In [33]:
#statistical summary of features
df.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [40]:
desc=df.describe()
both_zero = desc.columns[(desc.loc['min'] == 0) & (desc.loc['max'] == 0)].tolist()
print(both_zero)

[]


No columns where there are just zeroes, I searched for this so I can drop such a column

In [42]:
seventyfifth_zero = desc.columns[(desc.loc['min'] == 0) & (desc.loc['75%'] == 0)].tolist()
print(seventyfifth_zero)

['BsmtFinSF2', 'LowQualFinSF', 'BsmtHalfBath', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal']


### Dropping Sparse and Low-Signal Features

After analyzing the output of `df.describe()`, the following features were 
identified as sparse (75%+ zero values) or redundant and will be dropped or 
transformed:

- **BsmtFinSF2** — Represents the square footage of Type 2 finished basement area. 
Over 75% of values are 0, indicating most houses only have one basement type. 
Low predictive signal, dropping.

- **LowQualFinSF** — Represents low quality finished square footage across all floors. 
Over 75% of values are 0 as most houses in this dataset are standard quality builds. 
Quality information is already captured by `OverallQual` and `OverallCond`. Dropping, 
but will validate impact on model RMSE later.

- **BsmtHalfBath** — Represents the number of half bathrooms in the basement. 
seventyfifth_zero confirms 75%+ of values are 0, meaning most houses do not have 
a basement half bath. Sparse feature with low signal, dropping.

- **EnclosedPorch, 3SsnPorch, ScreenPorch** — All represent specific porch area types 
in square footage. Each individually has 75%+ zero values. Rather than dropping, 
these will be combined with `OpenPorchSF` and `WoodDeckSF` into a single binary 
feature `HasPorch` (1 if any porch/deck area > 0, else 0), reducing noise while 
retaining the information that a porch exists.

- **PoolArea** — Represents pool area in square footage. Over 75% of values are 0 
as most houses do not have a pool. Since `PoolQC` (pool quality) was already dropped 
earlier due to high missing values, retaining pool area adds no meaningful signal. 
Dropping.

- **MiscVal** — Represents the monetary value of miscellaneous features. Since 
`MiscFeature` was already dropped earlier due to 96% missing values, retaining 
the value of those features is redundant. Dropping.

In [43]:
df=df.drop(['BsmtFinSF2','LowQualFinSF','BsmtHalfBath','PoolArea','MiscVal'],axis=1)

In [51]:
porch_cols = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'WoodDeckSF']

df['HasPorch'] = (df[porch_cols].sum(axis=1) > 0).astype(int)
df=df.drop(porch_cols,axis=1)

In [52]:
df.sample(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,GarageArea,GarageQual,GarageCond,PavedDrive,MoSold,YrSold,SaleType,SaleCondition,SalePrice,HasPorch
468,469,20,RL,98.0,11428,Pave,IR1,Lvl,AllPub,Inside,...,866,TA,TA,Y,5,2007,WD,Normal,250000,1
161,162,60,RL,110.0,13688,Pave,IR1,Lvl,AllPub,Inside,...,726,TA,TA,Y,3,2008,WD,Normal,412500,1
1286,1287,20,RL,NaN,9790,Pave,Reg,Lvl,AllPub,Inside,...,528,TA,TA,Y,6,2010,WD,Normal,143000,1
804,805,20,RL,75.0,9000,Pave,Reg,Lvl,AllPub,Inside,...,286,TA,TA,Y,6,2006,WD,Family,118000,0
155,156,50,RL,60.0,9600,Pave,Reg,Lvl,AllPub,Corner,...,0,NaN,NaN,N,4,2008,WD,Normal,79000,1
